# Test Ingest (1 Day)
Pull 1 day from each data product, save to `cache/test_ingest_1day/`, then visualize from cache.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.abspath('.'))
from libs import GOESData, HRRRData, NDVIData, OrographyData, LandMaskData

EXTENT = (-118.615, -117.70, 33.60, 34.35)
DIM = 84
CACHE_DIR = 'cache/test_ingest_1day'
os.makedirs(CACHE_DIR, exist_ok=True)

START = '2024-08-01'
END   = '2024-08-02'

---
## Ingest

### GOES AOD / FRP / ADP

In [ ]:
goes_products = [
    ('goes_aod',       'ABI-L2-AODC', 'AOD'),
    ('goes_frp',       'ABI-L2-FDCC', 'Power'),
    ('goes_adp_smoke', 'ABI-L2-ADPC', 'Smoke'),
    ('goes_adp_dust',  'ABI-L2-ADPC', 'Dust'),
]

for name, product, subproduct in goes_products:
    out = os.path.join(CACHE_DIR, f'{name}.npy')
    if os.path.exists(out):
        print(f'{name}: already cached, skipping')
        continue
    print(f'Ingesting {name}...')
    lib_cache = os.path.join(CACHE_DIR, f'{name}_lib.npz')
    g = GOESData(
        start_date=f'{START} 00:00',
        end_date=f'{END} 00:00',
        extent=EXTENT, dim=DIM,
        product=product, subproduct=subproduct,
        cache_path=lib_cache,
        load_cache=os.path.exists(lib_cache),
        save_cache=True,
        verbose=True,
    )
    np.save(out, g.data)
    print(f'  Saved {name}: {g.data.shape}\n')

### HRRR

In [ ]:
hrrr_vars = [
    'temp_2m', 'dewpoint_2m', 'rh_2m', 'pressure_surface',
    'pressure_msl', 'pbl_height', 'smoke_massden', 'u_wind', 'v_wind',
]
all_cached = all(os.path.exists(os.path.join(CACHE_DIR, f'hrrr_{v}.npy')) for v in hrrr_vars)

if all_cached:
    print('HRRR: already cached, skipping')
else:
    print('Ingesting HRRR...')
    h = HRRRData(
        start_date=f'{START}-00',
        end_date=f'{END}-00',
        extent=EXTENT,
        grid_size=DIM,
        output_dir=os.path.join(CACHE_DIR, 'hrrr_work'),
        force_reprocess=True,
        verbose=True,
    )
    for var, arr in h.data.items():
        out = os.path.join(CACHE_DIR, f'hrrr_{var}.npy')
        np.save(out, arr)
        print(f'  Saved hrrr_{var}: {arr.shape}')

### MODIS NDVI

In [ ]:
ndvi_out = os.path.join(CACHE_DIR, 'ndvi.npy')
if os.path.exists(ndvi_out):
    print('NDVI: already cached, skipping')
else:
    print('Ingesting MODIS NDVI...')
    ndvi_work = os.path.join(CACHE_DIR, 'ndvi_work')
    os.makedirs(ndvi_work, exist_ok=True)
    n = NDVIData(
        start_date=START,
        end_date='2024-08-17',
        extent=EXTENT,
        dim=DIM,
        raw_dir=ndvi_work,
        save_dir=ndvi_work,
    )
    np.save(ndvi_out, n.data)
    print(f'  Saved ndvi: {n.data.shape}')

### Static Stubs

In [ ]:
T = 24
for name, Cls in [('orography', OrographyData), ('land_mask', LandMaskData)]:
    out = os.path.join(CACHE_DIR, f'{name}.npy')
    if os.path.exists(out):
        print(f'{name}: already cached, skipping')
        continue
    stub = Cls(extent=EXTENT, dim=DIM)
    tiled = np.tile(stub.data, (T, 1, 1))
    np.save(out, tiled)
    print(f'Saved {name}: {tiled.shape} (stub — all zeros)')

---
## Visualize from Cache

In [ ]:
def load(name):
    arr = np.load(os.path.join(CACHE_DIR, f'{name}.npy'))
    nan_pct = np.isnan(arr).mean() * 100
    print(f'{name}: shape={arr.shape}, NaN={nan_pct:.1f}%, '
          f'range=[{np.nanmin(arr):.4g}, {np.nanmax(arr):.4g}]')
    return arr

def show(data, titles, suptitle=None, cmap='viridis', t_idx=0):
    n = len(data)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1: axes = [axes]
    for ax, arr, title in zip(axes, data, titles):
        frame = arr[t_idx] if arr.ndim == 3 else arr
        im = ax.imshow(frame, cmap=cmap)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    plt.show()

### GOES

In [ ]:
goes_aod       = load('goes_aod')
goes_frp       = load('goes_frp')
goes_adp_smoke = load('goes_adp_smoke')
goes_adp_dust  = load('goes_adp_dust')

t = min(20, len(goes_aod) - 1)
show(
    [goes_aod, goes_frp, goes_adp_smoke, goes_adp_dust],
    ['AOD', 'FRP Power', 'ADP Smoke', 'ADP Dust'],
    suptitle=f'GOES  (frame {t})', t_idx=t
)

### HRRR

In [ ]:
hrrr = {v: load(f'hrrr_{v}') for v in hrrr_vars}

show(
    [hrrr['temp_2m'], hrrr['dewpoint_2m'], hrrr['rh_2m'],
     hrrr['pressure_surface'], hrrr['pressure_msl']],
    ['Temp 2m', 'Dewpoint 2m', 'RH 2m', 'P Sfc', 'P MSL'],
    suptitle='HRRR Thermodynamic  (frame 12)', t_idx=12
)

In [ ]:
show(
    [hrrr['pbl_height'], hrrr['u_wind'], hrrr['v_wind'], hrrr['smoke_massden']],
    ['PBL Height', 'U Wind', 'V Wind', 'Smoke'],
    suptitle='HRRR Dynamics & Smoke  (frame 12)', t_idx=12
)

### MODIS NDVI

In [ ]:
ndvi = load('ndvi')
show([ndvi], ['NDVI'], suptitle='MODIS NDVI', cmap='YlGn', t_idx=0)

### Static Stubs

In [ ]:
orog  = load('orography')
lmask = load('land_mask')
show(
    [orog, lmask],
    ['Orography (stub)', 'Land Mask (stub)'],
    suptitle='Static Fields — zeros until implemented', cmap='terrain'
)